In [1]:
!uv pip install pyngrok

Using Python 3.12.13 environment at: /usr
Checked 1 package in 56ms


In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [6]:
%cd "/content/drive/MyDrive/ML_in_production/NYC-Taxi"

/content/drive/MyDrive/ML_in_production/NYC-Taxi


In [2]:
from dotenv import load_dotenv
load_dotenv()

False

In [10]:
import os
from pyngrok import ngrok
from google.colab import userdata

# Try to get the key from the environment (.env)
ngrok_key = os.getenv("NGROK_KEY")

# If not found, try getting it from Colab Secrets
if not ngrok_key:
    try:
        ngrok_key = userdata.get("NGROK_KEY")
    except userdata.SecretNotFoundError:
        raise ValueError("NGROK_KEY is not set. Please add it to your .env file or Colab Secrets (🔑 on the left panel).")

ngrok.set_auth_token(ngrok_key)
print("ngrok auth token set successfully.")

ngrok auth token set successfully.


In [15]:
import subprocess
import time

# Start MLflow in the background using subprocess
subprocess.Popen(["uv", "run", "mlflow", "ui", "--host", "0.0.0.0", "--port", "5000"])

# Give the server a few seconds to start
time.sleep(3)

public_url = ngrok.connect(5000, host_header='rewrite').public_url
print(f"MLFlow live at: {public_url}")

MLFlow live at: https://interkinetic-cramponnae-rafaela.ngrok-free.dev


In [4]:
# Pass the Python variable 'public_url' to the shell environment
import os
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///mlflow.db"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "nyc-taxi-experiment"
os.environ['USERNAME'] = 'colab_user'

In [7]:
!uv run python3 src/pipelines/training.py run

root_path: /content/drive/MyDrive/ML_in_production/NYC-Taxi
Metaflow 2.19.22 executing Training for user:colab_user
Project: nyc_taxi, Branch: user.colab_user
Validating your flow...
    The graph looks good!
Running pylint...
    Pylint not found, so extra checks are disabled.
2026-04-13 02:35:40.420 Workflow starting (run-id 1776047739863940):
2026-04-13 02:35:42.303 [1776047739863940/start/1 (pid 86867)] Task is starting.
2026-04-13 02:35:47.135 [1776047739863940/start/1 (pid 86867)] root_path: /content/drive/MyDrive/ML_in_production/NYC-Taxi
2026-04-13 02:39:38.386 [1776047739863940/start/1 (pid 86867)] 2026-04-13 02:39:38,385 [INFO] Total rows in file: 4,251,015
2026-04-13 02:39:38.386 [1776047739863940/start/1 (pid 86867)] 2026-04-13 02:39:38,385 [INFO] Number of row groups: 5
2026-04-13 02:39:38.386 [1776047739863940/start/1 (pid 86867)] 2026-04-13 02:39:38,386 [INFO] Loading full dataset in 5 row groups...
2026-04-13 02:39:41.237 [1776047739863940/start/1 (pid 86867)] 2026-04-1

In [8]:
!uv run python3 src/serving/download_model.py

2026-04-13 03:16:43.213 | INFO     | __main__:validate_env:46 - Mlflow tracking URI: sqlite:///mlflow.db
2026-04-13 03:16:51.912 | INFO     | __main__:download_model:93 - ============================================================
2026-04-13 03:16:51.912 | INFO     | __main__:download_model:94 - ONNX Model download
2026-04-13 03:16:51.912 | INFO     | __main__:download_model:95 - ============================================================
2026-04-13 03:16:51.912 | INFO     | __main__:download_model:97 - Configuring MLflow....
2026-04-13 03:16:51.913 | INFO     | __main__:download_model:105 - Output directory: /content/drive/MyDrive/ML_in_production/NYC-Taxi/models/cache
2026-04-13 03:16:54.604 | INFO     | __main__:download_model:110 - 
Model: nyc-taxi-model
2026-04-13 03:16:54.604 | INFO     | __main__:download_model:111 - Requested version: latest
2026-04-13 03:16:54.791 | INFO     | __main__:download_model:125 - 	Resolved: version 1, run cd4d78d76d2342f79c610b04990f79e0
2026-04-13

In [19]:
!uv run uvicorn src.serving.api:app --reload

INFO:     Will watch for changes in these directories: ['/content/drive/MyDrive/ML_in_production/NYC-Taxi']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [47512] using WatchFiles
INFO:     Started server process [47520]
INFO:     Waiting for application startup.
2026-04-12 21:55:03.108 | INFO     | src.serving.api:lifespan:220 - Starting NYC Taxi Fare Prediction API...
2026-04-12 21:55:03.109 | INFO     | src.serving.api:lifespan:235 - Loading ONNX model from models/cache/model.onnx...
2026-04-12 21:55:03.148 | INFO     | src.serving.api:lifespan:241 -  Model loaded: (input: 'input')
2026-04-12 21:55:03.148 | INFO     | src.serving.api:lifespan:244 - Loading transformer from models/cache/transformer.joblib
2026-04-12 21:55:08.791 | INFO     | src.serving.api:lifespan:246 -  Transformer loaded: ColumnTransformer
2026-04-12 21:55:08.794 | INFO     | src.serving.api:lifespan:259 - 
2026-04-12 21:55:08.795 | INFO     | src.serv

KeyboardInterrupt: 

In [20]:
!curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"pickup_datetime":"2025-06-15 08:30:00","trip_distance":5.2}'

^C
